In [64]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import time


In [65]:
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless=new")  # nếu muốn chạy ẩn

# Chỉ rõ path tới chromedriver bạn vừa tải
driver = webdriver.Chrome(service=Service("chromedriver_win64/chromedriver.exe"), options=options)

driver.get("https://books.toscrape.com/")

In [66]:
# categories = driver.find_elements(By.CLASS_NAME, "side_categories")
# all_categories = categories[0].text.split("\n")[1:]  
# print(all_categories)

In [67]:
# pages = driver.find_element(By.CSS_SELECTOR, "li.next a")
# next_href = pages.get_attribute("href") if pages else None
# driver.get(next_href) if next_href else None

In [68]:
# books = driver.find_elements(By.CSS_SELECTOR, "ol.row a")
# for book in books:
#     book.click()

In [69]:
# # Book's information
# infos = driver.find_elements(By.CSS_SELECTOR, ".col-sm-6.product_main")
# for info in infos:
#     title = info.find_element(By.TAG_NAME, "h1").text
#     price = info.find_element(By.CLASS_NAME, "price_color").text
#     status = info.find_element(By.CSS_SELECTOR, ".instock.availability").text
#     rating = info.find_element(By.CSS_SELECTOR, "p.star-rating").get_attribute("class").split(" ")[-1]
#     rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
#     rating_num = rating_map[rating]
#     print(f"Title: {title}, Price: {price}, Status: {status}, Rating: {rating_num}")


In [70]:
# # Book's description 
# try:
#     description = driver.find_element(
#         By.XPATH, '//div[@id="product_description"]/following-sibling::p' #following-sibling::p tim the p dau tien xuat hien dang sau
#     ).text
# except:
#     description = "description is no available"
# print(description)

In [ ]:

books_data = []

while True:
    # lấy tất cả link sách ở trang hiện tại
    books = driver.find_elements(By.XPATH, "//h3/a")

    for book in books:
        link = book.get_attribute("href")

        # mở tab mới để lấy chi tiết sách
        driver.execute_script("window.open(arguments[0]);", link)
        driver.switch_to.window(driver.window_handles[1])

        try:
            title = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            ).text

            product_type = driver.find_element(By.XPATH, '//ul[@class="breadcrumb"]/li[3]/a').text

            price = driver.find_element(By.CLASS_NAME, "price_color").text

            try:
                desc_header = driver.find_element(By.ID, "product_description")
                desc = desc_header.find_element(By.XPATH, "following-sibling::p").text
            except NoSuchElementException:
                desc = "N/A"

            try:
                rating = driver.find_element(By.XPATH, "//p[contains(@class, 'star-rating')]").get_attribute("class").split()[-1]
            except NoSuchElementException:
                rating = "N/A"

            books_data.append({
                "title": title,
                "product_type":  product_type,
                "price": price,
                "description": desc,
                "rating": rating
            })
            print(f"✅ Lấy xong: {title}")

        except TimeoutException:
            print(f"⏳ Timeout khi load {link}")

        # đóng tab và quay lại trang danh sách
        driver.close()
        driver.switch_to.window(driver.window_handles[0])

    try:
        next_button = driver.find_element(By.XPATH, "//li[@class='next']/a")
        next_button.click()
    except NoSuchElementException:
        break  

driver.quit()

✅ Lấy xong: A Light in the Attic
✅ Lấy xong: Tipping the Velvet
✅ Lấy xong: Soumission
✅ Lấy xong: Sharp Objects
✅ Lấy xong: Sapiens: A Brief History of Humankind
✅ Lấy xong: The Requiem Red
✅ Lấy xong: The Dirty Little Secrets of Getting Your Dream Job
✅ Lấy xong: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
✅ Lấy xong: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
✅ Lấy xong: The Black Maria
✅ Lấy xong: Starving Hearts (Triangular Trade Trilogy, #1)
✅ Lấy xong: Shakespeare's Sonnets
✅ Lấy xong: Set Me Free
✅ Lấy xong: Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)
✅ Lấy xong: Rip it Up and Start Again
✅ Lấy xong: Our Band Could Be Your Life: Scenes from the American Indie Underground, 1981-1991
✅ Lấy xong: Olio
✅ Lấy xong: Mesaerion: The Best Science Fiction Stories 1800-1849
✅ Lấy xong: Libertarianism for Beginners
✅ Lấy xong: It's Only the Himalayas
✅ Lấy xong: In Her Wake
✅ Lấy xon

In [72]:
import pandas as pd

# books_data là list chứa các dict {"title":..., "price":..., "desc":..., "rating":...}
df = pd.DataFrame(books_data)

# Lưu ra file CSV
df.to_csv("books_data.csv", index=False, encoding="utf-8-sig")

print("Đã lưu dữ liệu vào books_data.csv")


Đã lưu dữ liệu vào books_data.csv
